In [220]:
import os
import io
import pyspark
import numpy as np
import matplotlib.pyplot as plt
import pyspark.sql.functions as F

from pyspark.sql import SparkSession, Window
from pyspark.conf import SparkConf
# from pyspark.context import SparkContext
from pyspark.sql.types import StringType, ArrayType, StructField, StructType, FloatType, DoubleType, IntegerType

from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, BucketedRandomProjectionLSH, VectorSlicer, StringIndexer, Imputer
from pyspark.ml.linalg import Vectors, VectorUDT

from concurrent.futures import ThreadPoolExecutor

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [233]:
DATA_DIR = "../include/data"

In [234]:
# # cloud
# URL = "abfss://{FOLDER_NAME}@sgppipelinesa.dfs.core.windows.net"
# SILVER_FOLDER_NAME = "sgppipelinesa-silver"
# SUB_FOLDER_NAME = "stage-02"
# SILVER_DATA_DIR = os.path.join(URL, "{SUB_FOLDER_NAME}").replace("\\", "/")
# SILVER_DATA_DIR

# local
SILVER_FOLDER_NAME = "silver"
SUB_FOLDER_NAME = "stage-03"
SILVER_DATA_DIR = os.path.join("{DATA_DIR}", "{FOLDER_NAME}", "{SUB_FOLDER_NAME}").replace("\\", "/")
SILVER_DATA_DIR

'{DATA_DIR}/{FOLDER_NAME}/{SUB_FOLDER_NAME}'

In [235]:
spark = SparkSession.builder.appName("app")\
    .config("spark.driver.memory", "16g")\
    .config("spark.executor.memory", "4g")\
    .config("spark.executor.cores", "2")\
    .config("spark.executor.instances", "3")\
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "100")\
    .getOrCreate()

# spark = SparkSession.builder.appName("app")\
#     .getOrCreate()

In [249]:
train_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "train_data.parquet"
    ).replace("\\", "/")
)

In [250]:
train_data_df.cache()

DataFrame[subjectId: string, freq_kurt: double, freq_skew: double, freq_entropy: double, freq_mean: double, freq_median: float, freq_mode: float, freq_min: float, freq_max: float, freq_var: double, freq_stddev: double, freq_first_quart: float, freq_third_quart: float, freq_range: float, freq_inter_quart_range: float, zcr: double, poly_feat_1: double, poly_feat_2: double, spec_cent: double, spec_bw: double, spec_flat: float, spec_roll: double, mel_spec_mean: float, mel_spec_median: float, mel_spec_mode: double, mel_spec_mode_cnt: double, mel_spec_min: float, mel_spec_max: float, mel_spec_range: float, mel_spec_var: float, mel_spec_std: float, mel_spec_first_quart: float, mel_spec_third_quart: float, mel_spec_inter_quart_range: float, mel_spec_entropy: float, mel_spec_kurt: float, mel_spec_skew: double, mel_spec_db_mean: float, mel_spec_db_median: float, mel_spec_db_mode: double, mel_spec_db_mode_cnt: double, mel_spec_db_min: float, mel_spec_db_max: float, mel_spec_db_range: float, mel_s

In [251]:
train_data_df.show()

+--------------------+------------------+------------------+-----------------+--------------------+-------------+-------------+-----------+---------+--------------------+-------------------+----------------+----------------+----------+----------------------+--------------------+--------------------+------------------+------------------+------------------+------------+------------------+-------------+---------------+--------------------+-----------------+------------+------------+--------------+------------+------------+--------------------+--------------------+--------------------------+----------------+-------------+------------------+----------------+------------------+-------------------+--------------------+---------------+---------------+-----------------+---------------+---------------+-----------------------+-----------------------+-----------------------------+-------------------+----------------+--------------------+---------+------------+-------------------+-------------+----

In [252]:
val_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "val_data.parquet"
    ).replace("\\", "/")
)

In [253]:
val_data_df.cache()

DataFrame[subjectId: string, freq_kurt: double, freq_skew: double, freq_entropy: double, freq_mean: double, freq_median: float, freq_mode: float, freq_min: float, freq_max: float, freq_var: double, freq_stddev: double, freq_first_quart: float, freq_third_quart: float, freq_range: float, freq_inter_quart_range: float, zcr: double, poly_feat_1: double, poly_feat_2: double, spec_cent: double, spec_bw: double, spec_flat: float, spec_roll: double, mel_spec_mean: float, mel_spec_median: float, mel_spec_mode: double, mel_spec_mode_cnt: double, mel_spec_min: float, mel_spec_max: float, mel_spec_range: float, mel_spec_var: float, mel_spec_std: float, mel_spec_first_quart: float, mel_spec_third_quart: float, mel_spec_inter_quart_range: float, mel_spec_entropy: float, mel_spec_kurt: float, mel_spec_skew: double, mel_spec_db_mean: float, mel_spec_db_median: float, mel_spec_db_mode: double, mel_spec_db_mode_cnt: double, mel_spec_db_min: float, mel_spec_db_max: float, mel_spec_db_range: float, mel_s

In [254]:
val_data_df.show()

+--------------------+------------------+--------------------+------------------+-------------------+-----------+----------+----------+----------+--------------------+-------------------+----------------+----------------+----------+----------------------+--------------------+--------------------+------------------+------------------+------------------+------------+------------------+-------------+---------------+--------------------+-----------------+------------+------------+--------------+------------+------------+--------------------+--------------------+--------------------------+----------------+-------------+------------------+----------------+------------------+-------------------+--------------------+---------------+---------------+-----------------+---------------+---------------+-----------------------+-----------------------+-----------------------------+-------------------+----------------+--------------------+---------+------------+-------------------+-------------+-------

In [255]:
test_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "test_data.parquet"
    ).replace("\\", "/")
)

In [256]:
test_data_df.cache()

DataFrame[subjectId: string, freq_kurt: double, freq_skew: double, freq_entropy: double, freq_mean: double, freq_median: float, freq_mode: float, freq_min: float, freq_max: float, freq_var: double, freq_stddev: double, freq_first_quart: float, freq_third_quart: float, freq_range: float, freq_inter_quart_range: float, zcr: double, poly_feat_1: double, poly_feat_2: double, spec_cent: double, spec_bw: double, spec_flat: float, spec_roll: double, mel_spec_mean: float, mel_spec_median: float, mel_spec_mode: double, mel_spec_mode_cnt: double, mel_spec_min: float, mel_spec_max: float, mel_spec_range: float, mel_spec_var: float, mel_spec_std: float, mel_spec_first_quart: float, mel_spec_third_quart: float, mel_spec_inter_quart_range: float, mel_spec_entropy: float, mel_spec_kurt: float, mel_spec_skew: double, mel_spec_db_mean: float, mel_spec_db_median: float, mel_spec_db_mode: double, mel_spec_db_mode_cnt: double, mel_spec_db_min: float, mel_spec_db_max: float, mel_spec_db_range: float, mel_s

In [257]:
test_data_df.show()

+--------------------+------------------+-------------------+------------------+--------------------+-------------+-------------+-----------+-----------+--------------------+--------------------+----------------+----------------+-----------+----------------------+--------------------+--------------------+-------------------+------------------+------------------+------------+------------------+-------------+---------------+--------------------+-----------------+------------+------------+--------------+------------+------------+--------------------+--------------------+--------------------------+----------------+-------------+-----------------+----------------+------------------+-------------------+--------------------+---------------+---------------+-----------------+---------------+---------------+-----------------------+-----------------------+-----------------------------+-------------------+----------------+-------------------+---------+-----------+-------------------+-------------+

# remove subjectId column from data frames

In [258]:
train_data_df = train_data_df.drop(*["subjectId"])

In [259]:
val_data_df = val_data_df.drop(*["subjectId"])

In [260]:
test_data_df = test_data_df.drop(*["subjectId"])

# impute potential nulls created during feature engineering

In [261]:
# Define the columns to impute
feat_list = list(filter(lambda col: not "label" in col, train_data_df.columns))
feat_list

['freq_kurt',
 'freq_skew',
 'freq_entropy',
 'freq_mean',
 'freq_median',
 'freq_mode',
 'freq_min',
 'freq_max',
 'freq_var',
 'freq_stddev',
 'freq_first_quart',
 'freq_third_quart',
 'freq_range',
 'freq_inter_quart_range',
 'zcr',
 'poly_feat_1',
 'poly_feat_2',
 'spec_cent',
 'spec_bw',
 'spec_flat',
 'spec_roll',
 'mel_spec_mean',
 'mel_spec_median',
 'mel_spec_mode',
 'mel_spec_mode_cnt',
 'mel_spec_min',
 'mel_spec_max',
 'mel_spec_range',
 'mel_spec_var',
 'mel_spec_std',
 'mel_spec_first_quart',
 'mel_spec_third_quart',
 'mel_spec_inter_quart_range',
 'mel_spec_entropy',
 'mel_spec_kurt',
 'mel_spec_skew',
 'mel_spec_db_mean',
 'mel_spec_db_median',
 'mel_spec_db_mode',
 'mel_spec_db_mode_cnt',
 'mel_spec_db_min',
 'mel_spec_db_max',
 'mel_spec_db_range',
 'mel_spec_db_var',
 'mel_spec_db_std',
 'mel_spec_db_first_quart',
 'mel_spec_db_third_quart',
 'mel_spec_db_inter_quart_range',
 'mel_spec_db_entropy',
 'mel_spec_db_kurt',
 'mel_spec_db_skew',
 'mfcc_mean',
 'mfcc_median

In [262]:
# Create output column names (e.g., by appending '_imp')
output_cols = [col + "_imp" for col in feat_list]

In [263]:
# Create an Imputer instance
imputer = Imputer(
    inputCols=feat_list,
    outputCols=output_cols,
    strategy="mean"
)

In [264]:
# Fit the Imputer model to the DataFrame and transform it
model = imputer.fit(train_data_df)
train_data_df_imp = model.transform(train_data_df)

In [265]:
train_data_df_imp = train_data_df_imp.drop(*feat_list)

In [266]:
# Show the result
train_data_df_imp.show()

+-----+------------------+------------------+-----------------+--------------------+---------------+-------------+------------+------------+--------------------+-------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+--------------

In [267]:
# Fit the Imputer model to the DataFrame and transform it
model = imputer.fit(val_data_df)
val_data_df_imp = model.transform(val_data_df)

In [268]:
val_data_df_imp = val_data_df_imp.drop(*feat_list)

In [269]:
# Show the result
val_data_df_imp.show()

+-----+------------------+--------------------+------------------+-------------------+---------------+-------------+------------+------------+--------------------+-------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+------------

In [270]:
# Fit the Imputer model to the DataFrame and transform it
model = imputer.fit(test_data_df)
test_data_df_imp = model.transform(test_data_df)

In [271]:
test_data_df_imp = test_data_df_imp.drop(*feat_list)

In [272]:
# Show the result
test_data_df_imp.show()

+-----+------------------+-------------------+------------------+--------------------+---------------+-------------+------------+------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+-----------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+-----------

# convert the label categorical columns to a numerical value, 0 for male and 1 for female 

In [273]:
label_cond = F.when(
    F.col("label") == "male",
    0
).otherwise(1)

In [302]:
train_data_df_imp = train_data_df_imp.withColumn("label", label_cond)
train_data_df_imp.show()

+-----+------------------+------------------+-----------------+--------------------+---------------+-------------+------------+------------+--------------------+-------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+--------------

In [303]:
val_data_df_imp = val_data_df_imp.withColumn("label", label_cond)
val_data_df_imp.show()

+-----+------------------+--------------------+------------------+-------------------+---------------+-------------+------------+------------+--------------------+-------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+------------

In [304]:
test_data_df_imp = test_data_df_imp.withColumn("label", label_cond)
test_data_df_imp.show()

+-----+------------------+-------------------+------------------+--------------------+---------------+-------------+------------+------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+-----------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+-----------

In [305]:
train_data_df.count(), val_data_df.count(), test_data_df.count()

(213111, 47500, 54036)

# how string index works

In [306]:
toy_df = train_data_df_imp.sample(withReplacement=False, fraction=0.5)
toy_df.show()

+-----+------------------+-------------------+------------------+--------------------+---------------+-------------+------------+------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+-----------

In [307]:
toy_df = toy_df.withColumn("some_cat_col", F.lit("some literal"))
toy_df.show()

+-----+------------------+-------------------+------------------+--------------------+---------------+-------------+------------+------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+-----------

In [308]:
num_cols = list(filter(lambda col: not "label" in col, train_data_df_imp.columns))
num_cols

['freq_kurt_imp',
 'freq_skew_imp',
 'freq_entropy_imp',
 'freq_mean_imp',
 'freq_median_imp',
 'freq_mode_imp',
 'freq_min_imp',
 'freq_max_imp',
 'freq_var_imp',
 'freq_stddev_imp',
 'freq_first_quart_imp',
 'freq_third_quart_imp',
 'freq_range_imp',
 'freq_inter_quart_range_imp',
 'zcr_imp',
 'poly_feat_1_imp',
 'poly_feat_2_imp',
 'spec_cent_imp',
 'spec_bw_imp',
 'spec_flat_imp',
 'spec_roll_imp',
 'mel_spec_mean_imp',
 'mel_spec_median_imp',
 'mel_spec_mode_imp',
 'mel_spec_mode_cnt_imp',
 'mel_spec_min_imp',
 'mel_spec_max_imp',
 'mel_spec_range_imp',
 'mel_spec_var_imp',
 'mel_spec_std_imp',
 'mel_spec_first_quart_imp',
 'mel_spec_third_quart_imp',
 'mel_spec_inter_quart_range_imp',
 'mel_spec_entropy_imp',
 'mel_spec_kurt_imp',
 'mel_spec_skew_imp',
 'mel_spec_db_mean_imp',
 'mel_spec_db_median_imp',
 'mel_spec_db_mode_imp',
 'mel_spec_db_mode_cnt_imp',
 'mel_spec_db_min_imp',
 'mel_spec_db_max_imp',
 'mel_spec_db_range_imp',
 'mel_spec_db_var_imp',
 'mel_spec_db_std_imp',
 'm

In [309]:
assembler = VectorAssembler(inputCols=num_cols, outputCol="features")
# assembler.setHandleInvalid("keep")

In [310]:
cat_cols = ["some_cat_col"]

In [311]:
target_col = "label"

In [312]:
list(set(cat_cols) - set(["label"]))

['some_cat_col']

In [313]:
# index the string cols, except possibly for the label col
index_suffix = "_index"
cat_cols_to_vectorize = list(set(cat_cols) - set([target_col]))
assemble_stages = [
    StringIndexer(inputCol=column, outputCol=column + index_suffix).fit(toy_df) 
    for column in cat_cols
]

In [314]:
# add the stage of numerical vector assembler
assemble_stages.append(assembler)

#### note that string indexers, scalers, assemblers rely heavily on its columsn not being null so make sure all null values are imputed first in order to avoid `SparkException: Values to assemble cannot be null`  

In [315]:
assemble_stages

[StringIndexerModel: uid=StringIndexer_9d71a233784d, handleInvalid=error,
 VectorAssembler_b2bb5ce3d1e6]

In [316]:

pipeline = Pipeline(stages=assemble_stages)
model = pipeline.fit(train_data_df)

In [318]:
toy_df_vec = model.transform(toy_df)

In [319]:
toy_df_vec.show()

+-----+------------------+-------------------+------------------+--------------------+---------------+-------------+------------+------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+--------------------+---------------------+----------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+-----------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+---------------------------------+-----------

In [320]:
# drop original num cols and cat cols
drop_cols = num_cols + cat_cols
keep_cols = [a for a in toy_df_vec.columns if a not in drop_cols]
keep_cols

['label', 'some_cat_col_index', 'features']

In [321]:
vectorized = toy_df_vec.select(*keep_cols) \
.withColumn('label', toy_df_vec[target_col])
vectorized.show()

+-----+------------------+--------------------+
|label|some_cat_col_index|            features|
+-----+------------------+--------------------+
|    0|               0.0|[9.25783466547273...|
|    0|               0.0|[9.10845109950064...|
|    0|               0.0|[8.95897453398126...|
|    0|               0.0|[8.56461421405954...|
|    0|               0.0|[9.39443329421345...|
|    0|               0.0|[10.8089818711146...|
|    0|               0.0|[12.1258356114221...|
|    0|               0.0|[13.3033969612619...|
|    0|               0.0|[20.9624989252575...|
|    0|               0.0|[20.8216981532022...|
|    0|               0.0|[20.4388561246142...|
|    0|               0.0|[20.0631937667648...|
|    0|               0.0|[20.0542440132208...|
|    0|               0.0|[5.01977464305526...|
|    0|               0.0|[4.72190908429700...|
|    0|               0.0|[5.86308695052849...|
|    0|               0.0|[5.68305112030742...|
|    0|               0.0|[7.02945670330

In [322]:
vectorized.count()

107079

In [323]:
vectorized.cache()

DataFrame[label: int, some_cat_col_index: double, features: vector]

In [324]:
min_class = vectorized.where(F.col("label") == 1)
maj_class = vectorized.where(F.col("label") == 0)
min_class

DataFrame[label: int, some_cat_col_index: double, features: vector]

In [325]:
min_class.show()

+-----+------------------+--------------------+
|label|some_cat_col_index|            features|
+-----+------------------+--------------------+
|    1|               0.0|[7.84652323400643...|
|    1|               0.0|[7.61175327001911...|
|    1|               0.0|[8.31154163778706...|
|    1|               0.0|[8.02559510258002...|
|    1|               0.0|[10.8951283964650...|
|    1|               0.0|[10.1890205296851...|
|    1|               0.0|[8.29750916184377...|
|    1|               0.0|[7.90890991126470...|
|    1|               0.0|[7.42949087916549...|
|    1|               0.0|[6.87635997841680...|
|    1|               0.0|[6.11438381484787...|
|    1|               0.0|[5.69478836661830...|
|    1|               0.0|[5.52272352879086...|
|    1|               0.0|[5.48483507851578...|
|    1|               0.0|[5.40451707943412...|
|    1|               0.0|[4.93250136131935...|
|    1|               0.0|[4.04836845586922...|
|    1|               0.0|[3.08170067154